# Svon / Svon-CGW — Ücretsiz Kaggle `GPU T4 x2` (32 GB VRAM) Bulut Çalıştırıcısı

Bu defter, yerel RTX 4060 8 GB üzerindeki **Epoch I** (`Pristine-50`, `Dev-500 GRPO`, `Q201..Q500`) doğrulamasını hiç bölmeden; ağır **Epoch II (`Svon / Burak` Gerçek Cımbız Kimlik Cerrahisi)** ve **Epoch III (`Svon-CGW` Holomorphic CVNN-DeltaNet + 2-Hop Sheaf-GNN)** adımlarını **Kaggle `T4 x2` (`2 × 16 GB = 32 GB VRAM`)** üzerinde **kayıpsız FP16 (`device_map="auto"`)** olarak çalıştırır.

### Kaggle Ayarları (Çalıştırmadan Önce)
- **Accelerator:** `GPU T4 x2`
- **Internet:** Açık (`Qwen/Qwen3.5-9B` ve `transformers >= 5.16.0` kurulumu için) — veya ağırlık + `sft/` dosyalarını Kaggle Dataset olarak `/kaggle/input/` içine bağlayın.

In [ ]:
import os, sys, torch
print("PyTorch:", torch.__version__, "| GPUs:", torch.cuda.device_count())
assert torch.cuda.is_available() and torch.cuda.device_count() >= 1, (
    "Kaggle GPU aktif değil (GPUs=0)! Hesapta SMS Telefon Doğrulaması "
    "(https://www.kaggle.com/settings -> Phone Verification) yapılmamışsa Kaggle GPU ve İnternet erişimini kapatır."
)
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} ({props.total_memory / 1024**3:.2f} GiB)")
!nvidia-smi
!pip uninstall -y -q torchao
!pip install -q -U "transformers>=5.16.0" "peft>=0.15.0" "safetensors>=0.4.5" "accelerate>=1.0.0" "bitsandbytes>=0.45.0" "jinja2>=3.1.6"

In [ ]:
import glob, os, shutil, sys

# `/kaggle/input` veya mevcut dizin altında `svon_alphaedit_surgeon.py` arar
SEARCH_ROOTS = ["/kaggle/working", "/kaggle/input", os.getcwd()]
SFT_CODE_DIR = None
for root in SEARCH_ROOTS:
    if os.path.exists(root):
        hits = glob.glob(os.path.join(root, "**", "svon_alphaedit_surgeon.py"), recursive=True)
        if hits:
            SFT_CODE_DIR = os.path.dirname(hits[0])
            break

assert SFT_CODE_DIR is not None, (
    "`sft/` kodları bulunamadı! Lütfen `cloud_runtime_utils.py`, `svon_*.py`, `eval_svon_*.py`, "
    "`train_svon_cgw_architecture.py` ve `run_kaggle_svon_epoch2_epoch3.py` dosyalarını Kaggle Dataset olarak ekleyin."
)

# Yazılabilir çalışma kopyası (`/kaggle/working/sft_run`)
WORK_SFT = "/kaggle/working/sft_run" if os.path.exists("/kaggle") else os.path.abspath(SFT_CODE_DIR)
if os.path.abspath(SFT_CODE_DIR) != os.path.abspath(WORK_SFT):
    shutil.copytree(SFT_CODE_DIR, WORK_SFT, dirs_exist_ok=True)
if WORK_SFT not in sys.path:
    sys.path.insert(0, WORK_SFT)

from run_kaggle_svon_epoch2_epoch3 import resolve_default_model_path
MODEL_PATH = resolve_default_model_path(online=True)
# Özel model yolu vermek isterseniz üstüne yazın (örn. Kendi SFT/GRPO model klasörünüz):
# MODEL_PATH = "/kaggle/input/your-qwen35-9b-dataset"

print("WORK_SFT:  ", WORK_SFT)
print("MODEL_PATH:", MODEL_PATH)

In [ ]:
# GPU'ya geçmeden önce matematik & SafeTensors / .qmem birim testlerini doğrula
import os, subprocess, sys
tests_dir = os.path.join(WORK_SFT, "tests", "prod_suite")
if os.path.exists(tests_dir):
    for t_file in ("test_svon_alphaedit_surgeon.py", "test_svon_cgw_hybrid_gnn_cvnn.py"):
        tp = os.path.join(tests_dir, t_file)
        if os.path.exists(tp):
            print(f"Running {t_file}...")
            subprocess.run([sys.executable, "-B", "-m", "unittest", tp, "-v"], check=True)

In [ ]:
# EPOCH II: Tweezer SwiGLU Trace -> AlphaEdit Yol A LoRA -> Zero-Prompt & 3. Şahıs Doğrulama
# T4 x2 (32 GB) üzerinde `--fp16` ile nicemleme kaybı olmadan çalışır.
# Geçerse Yol B FP16 yamasını da açmak için `--export_path_b_on_pass` ekleyin.
import os, subprocess, sys
cmd_ep2 = [
    sys.executable, "-B", os.path.join(WORK_SFT, "run_kaggle_svon_epoch2_epoch3.py"),
    "--stage", "epoch2",
    "--model_path", MODEL_PATH,
    "--work_dir", "/kaggle/working/svon_out",
    "--fp16", "--online",
    "--top_layers", "8", "--neurons_per_layer", "256",
    "--min_ratio", "1.25", "--max_rel_frob", "0.014", "--max_rank", "24",
]
subprocess.run(cmd_ep2, check=True)

In [ ]:
# EPOCH III: Svon-CGW (24 Holomorphic CVNN-DeltaNet + 8 2-Hop Sheaf-GNN)
# Çıktılar: `svon_cgw_hybrid.safetensors` (Python sodals_serve.py) + `svon_cgw_hybrid.qmem` (Rust motoru)
import os, subprocess, sys
cmd_ep3 = [
    sys.executable, "-B", os.path.join(WORK_SFT, "run_kaggle_svon_epoch2_epoch3.py"),
    "--stage", "epoch3",
    "--model_path", MODEL_PATH,
    "--work_dir", "/kaggle/working/svon_out",
    "--fp16", "--online",
    "--cgw_steps", "50", "--cgw_lr", "1e-3",
    "--cgw_phase_rank", "16", "--cgw_phase_loss_weight", "0.25",
    "--out_zip", "/kaggle/working/svon_kaggle_artifacts.zip",
]
subprocess.run(cmd_ep3, check=True)

In [ ]:
import json, os, zipfile
zip_p = "/kaggle/working/svon_kaggle_artifacts.zip"
if os.path.exists(zip_p):
    print(f"İndirilecek Paket Hazır: {zip_p} ({os.path.getsize(zip_p)/1024/1024:.2f} MB)")
    with zipfile.ZipFile(zip_p, "r") as zf:
        for name in sorted(zf.namelist()):
            print("  -", name)

summary_p = "/kaggle/working/svon_out/cloud_pipeline_summary.json"
if os.path.exists(summary_p):
    print("\nÖzet:")
    print(json.dumps(json.load(open(summary_p, encoding="utf-8")), indent=2, ensure_ascii=False))

print("\nYerel Windows makinenize indirdikten sonra canlı çalıştırma:")
print("  python sft/sodals_serve.py --cli --svon_identity_lora sft/qwen_lora_adapter_svon_alphaedit --enable_svon_cgw --svon_cgw_weights sft/svon_cgw_checkpoints/svon_cgw_hybrid.safetensors")